# CASMI 2026 — corrected de novo experiment

Derived from [inversion's CASMI tutorial](https://www.kaggle.com/code/inversion/casmi-denovo-tutorial-notebook), downloaded version 9. This is an **untrained experimental notebook**, not a demonstrated leaderboard improvement. It fixes same-token target leakage and missing generation history, adds decoder positions, uses tautomer-aware MRR@25, and restores the best checkpoint.

Attach the official competition data. Select one Kaggle GPU and disable Internet. In Kaggle Dependency Manager provision `lightning`, `tokenizers`, `pyarrow`, `pandas`, `numpy`, and **`rdkit==2026.3.3`** before the offline commit. This notebook does not download anything. Runtime and GPU memory still need measurement on Kaggle; the training timer alone does not guarantee the nine-hour total limit.

The separate retrieval notebook produces the delivered CSV. This notebook trains a different model and must be validated before replacing that submission. Do not load the original tutorial's weights into the corrected architecture.

**Supplied-data overlap found:** all 1,213 test spectra (400 molecules) match the `enveda-180` training source after preprocessing. All retrieval top guesses agree with the unique matching reference connectivities. This is not a hidden-test score. The original tutorial excludes that source; its reason was not established. See the bundled overlap audit. Structure-disjoint validation remains essential for assessing de novo generalization.

In [ ]:
import os, time, math, random, json, hashlib
from pathlib import Path
from functools import lru_cache, partial
from collections import defaultdict
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
from torch import nn
import torch.nn.functional as F
from torch.nn import TransformerEncoderLayer, TransformerEncoder
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from tokenizers import Tokenizer, models, trainers
from tokenizers.processors import TemplateProcessing
import lightning as L
from rdkit import Chem, rdBase, RDLogger
from rdkit.Chem.MolStandardize import rdMolStandardize
from tqdm.auto import tqdm
tqdm.pandas()
SEED = 42
L.seed_everything(SEED, workers=True)
RDLogger.DisableLog('rdApp.*')
assert rdBase.rdkitVersion == '2026.03.3', 'Provision rdkit==2026.3.3 before committing offline'
assert torch.cuda.is_available(), 'This training notebook requires a GPU'
START = time.monotonic()
PRECISION = 'bf16-mixed' if torch.cuda.is_bf16_supported() else '16-mixed'
TAUTOMER = rdMolStandardize.TautomerEnumerator()
@lru_cache(maxsize=350000)
def identity(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None: return None
        mol = TAUTOMER.Canonicalize(mol)
        key = Chem.MolToInchiKey(mol).split('-')[0]
        return (key, Chem.MolToSmiles(mol, isomericSmiles=False)) if len(key)==14 else None
    except Exception: return None

roots = [Path('/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra'),
         Path('/kaggle/input/enveda-CASMI26-molecule-id-mass-spectra')]
COMP_DIR = next((p for p in roots if (p/'train.parquet').exists()), None)
if COMP_DIR is None: raise FileNotFoundError('Attach official competition data; adjust roots if needed')
WORKING_DIR = Path('/kaggle/working')
WORKING_DIR.mkdir(exist_ok=True)
TRAIN_PATH, TEST_PATH = COMP_DIR/'train.parquet', COMP_DIR/'test.parquet'
SAMPLE_SUBMISSION_PATH = COMP_DIR/'sample_submission.csv'
MAX_LEN, MAX_SMILES_TOKENS = 128, 160
BPE_PAD_ID, BPE_BOS_ID, BPE_EOS_ID = 0, 1, 2
VOCAB_SIZE, BATCH_SIZE, NUM_WORKERS = 512, 64, 2
PRECURSOR_INTENSITY, MZ_PAD_ID, INTENSITY_PAD_ID = 2., 0, 0
MAX_TRAIN_SPECTRA, N_VAL_STRUCTURES = 400000, 100


## Structure-disjoint split and bounded data loading
Split on tautomer-canonical connectivity, not spectrum rows. All provided training sources are eligible. Every spectrum of a validation connectivity is excluded from training. One validation spectrum per structure keeps the per-epoch cost bounded; this is an approximate validation setting, not the hidden test's multi-spectrum distribution.

In [ ]:
reference = pq.ParquetFile(TRAIN_PATH)
unique_smiles = set()
for block in reference.iter_batches(batch_size=100000, columns=['normalized_smiles']):
    unique_smiles.update(block.column(0).to_pylist())
canonical = {s: identity(s) for s in tqdm(sorted(s for s in unique_smiles if isinstance(s,str)))}
keys = sorted({v[0] for v in canonical.values() if v})
rng = np.random.default_rng(SEED)
val_keys = set(rng.choice(keys, min(N_VAL_STRUCTURES, len(keys)//10), replace=False))
train_parts, val_parts, seen_val = [], [], set()
cols = ['normalized_smiles', 'precursor_mz', 'ms2_mzs', 'ms2_normalized_intensities']
# Uniform Bernoulli row sample, capped after loading; no test data used.
fraction = min(1., MAX_TRAIN_SPECTRA * 1.15 / reference.metadata.num_rows)
for block in reference.iter_batches(batch_size=32768, columns=cols):
    df = block.to_pandas()
    df['identity'] = df.normalized_smiles.map(canonical)
    df = df[df.identity.notna()].copy()
    df['key'] = df.identity.map(lambda x:x[0])
    df['normalized_smiles'] = df.identity.map(lambda x:x[1])
    is_val = df.key.isin(val_keys)
    val = df[is_val & ~df.key.isin(seen_val)].drop_duplicates('key')
    seen_val.update(val.key)
    val_parts.append(val.drop(columns=['identity']))
    train = df[~is_val]
    train_parts.append(train.loc[rng.random(len(train)) < fraction].drop(columns=['identity']))
casmi_train_df = pd.concat(train_parts, ignore_index=True)
casmi_val_df = pd.concat(val_parts, ignore_index=True)
del train_parts, val_parts, canonical, unique_smiles
if len(casmi_train_df)>MAX_TRAIN_SPECTRA:
    casmi_train_df=casmi_train_df.sample(MAX_TRAIN_SPECTRA, random_state=SEED).reset_index(drop=True)
assert not set(casmi_train_df.key) & set(casmi_val_df.key)
assert len(casmi_train_df) and len(casmi_val_df)
print('Train spectra:',len(casmi_train_df),'validation structures:',len(casmi_val_df))

def process_spectra(df):
    df = df.copy()
    mzs, intensities = [], []
    for row in df.itertuples():
        mz = np.asarray(row.ms2_mzs, dtype=np.float32)
        it = np.asarray(row.ms2_normalized_intensities, dtype=np.float32)
        if len(mz)!=len(it): raise ValueError('Peak arrays differ in length')
        ok = np.isfinite(mz)&np.isfinite(it)&(mz>0)&(it>0)
        mz,it=mz[ok],it[ok]
        order=np.argsort(it)[::-1][:MAX_LEN-1]
        mz,it=mz[order],it[order]
        it=it/max(float(it.max()),1e-12) if len(it) else it
        mzs.append(mz); intensities.append(it)
    df['processed_mzs'],df['processed_intensities']=mzs,intensities
    return df
casmi_train_df=process_spectra(casmi_train_df)
casmi_val_df=process_spectra(casmi_val_df)


In [ ]:
def train_bpe_tokenizer(structures):
    os.environ["TOKENIZERS_PARALLELISM"] = "true"     # turn on tokenizers parallelism to speed up bpe training
    
    full_alphabet = sorted(set("".join(structures) + "BCNOPSFIbcnops[]=#()1234567890+-/@\\.%H"))
    tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
    special_tokens = [
        ("<pad>", BPE_PAD_ID),
        ("<s>", BPE_BOS_ID),
        ("</s>", BPE_EOS_ID),
    ]
    tokenizer.post_processor = TemplateProcessing(
        single="<s> $A </s>", special_tokens=special_tokens
    )
    special_tokens = ["<pad>", "<s>", "</s>", "<unk>"]
    
    trainer = trainers.BpeTrainer(
        vocab_size=VOCAB_SIZE,
        initial_alphabet=full_alphabet,
        special_tokens=special_tokens,
        show_progress=True,
    )
    train_iterator = structures
    tokenizer.train_from_iterator(train_iterator, trainer)
    
    os.environ["TOKENIZERS_PARALLELISM"] = "false"     # turn off tokenizers parallelism
    return tokenizer

def tokenize_df_smiles(df, tokenizer):
    df['bpe_tokenized_smiles'] = df.normalized_smiles.progress_apply(
        lambda x: np.array(tokenizer.encode(x).ids, dtype=int)
    )
    return df

## tokenize smiles
print('training bpe...')
bpe_train_structures = list(casmi_train_df.normalized_smiles.unique())
tokenizer = train_bpe_tokenizer(bpe_train_structures)

print('processing smiles...')
casmi_train_df = tokenize_df_smiles(casmi_train_df, tokenizer=tokenizer)
casmi_val_df = tokenize_df_smiles(casmi_val_df, tokenizer=tokenizer)

# utility for decoding bpe smiles
def decode_tokenized_smiles(bpe_smiles_tokens, tokenizer=tokenizer):
    return tokenizer.decode(bpe_smiles_tokens).replace(' ', '')
tokenizer.save(str(WORKING_DIR/'tokenizer.json'))
for frame in [casmi_train_df, casmi_val_df]:
    too_long=frame.bpe_tokenized_smiles.map(len)>MAX_SMILES_TOKENS
    print('Long sequences excluded:',int(too_long.sum()))
    frame.drop(frame.index[too_long],inplace=True)
assert len(casmi_train_df) and len(casmi_val_df)


In [ ]:
class PandasDataset(Dataset):
    def __init__(self, df, column_list, shuffle=True):
        # Prediction keeps the original row order so rows stay aligned with their molecule_id.
        self.df = df[column_list].sample(frac=1, random_state=0) if shuffle else df[column_list]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        return self.df.iloc[idx].to_dict()

In [ ]:
# define datamodule class
class CASMIDataModule(L.LightningDataModule):
    def __init__(self, batch_size, predict_batch_size=None):
        super().__init__()
        self.batch_size = batch_size
        # Generation expands the batch by n_samples, so prediction needs a far smaller batch than
        # training. See the predict cell for the arithmetic.
        self.predict_batch_size = predict_batch_size or batch_size

    def setup(self, stage):
        columns = ['precursor_mz', 'processed_mzs', 'processed_intensities', 'bpe_tokenized_smiles', 'normalized_smiles']
        if stage == "fit":
            self.train_df = PandasDataset(casmi_train_df, column_list=columns)
            self.val_df = PandasDataset(casmi_val_df, column_list=columns)
        elif stage == "validate":
            self.val_df = PandasDataset(casmi_val_df, column_list=columns)
        elif stage == "predict":
            # test.parquet carries no labels, so only the spectrum columns exist. molecule_id rides
            # along because scoring is per molecule, not per spectrum, and a molecule's spectra have
            # to be grouped back together afterwards.
            self.predict_df = PandasDataset(
                casmi_test_df,
                column_list=['molecule_id', 'precursor_mz', 'processed_mzs', 'processed_intensities'],
                shuffle=False,
            )

    def train_dataloader(self):
        return DataLoader(
            self.train_df,
            batch_size=self.batch_size,
            collate_fn=self.get_collator('fit'),
            num_workers=NUM_WORKERS,
            persistent_workers=NUM_WORKERS > 0,
            pin_memory=True,
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_df,
            batch_size=2,
            collate_fn=self.get_collator('validate'),
            num_workers=NUM_WORKERS,
            persistent_workers=NUM_WORKERS > 0,
            pin_memory=True,
        )

    def predict_dataloader(self):
        return DataLoader(
            self.predict_df,
            batch_size=self.predict_batch_size,
            collate_fn=self.get_collator('predict'),
            num_workers=NUM_WORKERS,
            persistent_workers=NUM_WORKERS > 0,
            pin_memory=True,
        )

    def get_collator(self, stage):
        return partial(self._collator, stage=stage)

    def _collator(self, data, stage='fit'):
        mzs = [[row['precursor_mz']] + row['processed_mzs'].tolist() for row in data]  # prepend precursor to m/zs
        ints = [[PRECURSOR_INTENSITY] + row['processed_intensities'].tolist() for row in data]  # prepend 2. to intensities
        if stage == 'fit' or stage == 'validate':
            labels = [row['bpe_tokenized_smiles'] for row in data]

        # collate peaks
        max_peak_len = max([len(x) for x in mzs])
        mzs = [list(x) + [MZ_PAD_ID] * (max_peak_len - len(x)) for x in mzs]
        ints = [list(x) + [INTENSITY_PAD_ID] * (max_peak_len - len(x)) for x in ints]
        mz_array = torch.tensor(mzs, dtype=torch.float32)
        intensity_array = torch.tensor(ints, dtype=torch.float32)
        attention_mask = torch.where(mz_array == MZ_PAD_ID, 0, 1)

        # collate labels
        batch = {}
        if stage == 'fit' or stage == 'validate':
            max_smiles_len = max([len(x) for x in labels])
            labels = [list(x) + [BPE_PAD_ID] * (max_smiles_len - len(x)) for x in labels]
            label_array = torch.tensor(labels, dtype=torch.long)

            batch['mzs'] = mz_array
            batch['intensities'] = intensity_array
            batch['attention_mask'] = attention_mask
            batch['structure_tokens'] = label_array # tokenized labels for training
        if stage == 'validate':
            batch['smiles'] = [row['normalized_smiles'] for row in data] # include original smiles for validation metrics
        else:
            batch = batch | {
                'mzs': mz_array,
                'intensities': intensity_array,
                'attention_mask': attention_mask,
            }
        if stage == 'predict':
            batch['molecule_id'] = [row['molecule_id'] for row in data]
        return batch

# define datamodule
datamodule = CASMIDataModule(batch_size=BATCH_SIZE)
datamodule.setup('fit')

## Corrected autoregressive model
Training uses `tokens[:, :-1] → tokens[:, 1:]`. Every decoder call uses a causal mask and absolute position embeddings. Generation passes the complete prefix, freezes finished sequences, disallows special-token sampling, and rejects unfinished sequences. This implementation has no KV cache; generation is intentionally batched conservatively.

In [ ]:
# define module classes

### peak embedder
class PeakEmbedder(nn.Module):
    """embed (m/z, intensity) peak pairs with sinusoidal embeddings from Voronov et al"""
    def __init__(self, d_model, dropout, sin_dim=None, mz_log_lims=(-2., 3.), mz_log_power=1.0):
        super().__init__()
        sin_dim = sin_dim if sin_dim is not None else d_model
        self.dropout = dropout

        wavelength = torch.pow(
            10,
            (mz_log_lims[1] - mz_log_lims[0]) * torch.pow(
                torch.linspace(0, 1, int(sin_dim / 2)), 
                mz_log_power,
            ) + mz_log_lims[0],
        )
        frequency = 2 * np.pi / wavelength
        self._frequency = nn.Parameter(frequency, requires_grad=False)
        self._ff_block_1 = nn.Sequential(*[
            nn.Linear(sin_dim, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model),
            nn.Dropout(dropout),
        ])
        self._ff_block_2 = nn.Sequential(*[
            nn.Linear(d_model+1, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model),
            nn.Dropout(dropout),
        ])

    def forward(self, mz_tensor, intensity_tensor):
        # "mz embedding" embeds just the m/z value
        omega_mz = self._frequency.view(
            *(1 for _ in range(mz_tensor.ndim)), -1
        ) * mz_tensor.unsqueeze(-1)
        sin = torch.sin(omega_mz)
        cos = torch.cos(omega_mz)
        mz_vecs = torch.cat([sin, cos], dim=-1)
        mz_embeds = self._ff_block_1(mz_vecs)

        # "peak embedding" embeds both the m/z and intensity
        peak_embeds = torch.cat([mz_embeds, intensity_tensor.unsqueeze(2)], dim=2)
        return self._ff_block_2(peak_embeds)

        
### spectrum encoder
class SpectrumEncoder(nn.Module):
    def __init__(self, embed_dim, n_heads, n_layers, dim_feedforward=None, dropout=0.1, activation='relu'):
        super().__init__()
        dim_feedforward = dim_feedforward if dim_feedforward is not None else 4 * embed_dim
        self.encoder = TransformerEncoder(
            TransformerEncoderLayer(
                embed_dim,
                n_heads,
                dim_feedforward=dim_feedforward,
                batch_first=True,
                dropout=dropout,
                activation=activation,
            ),
            n_layers,
        )
        self.init_weights()

    def forward(self, sequence_input, attention_mask):
        pad_mask = (attention_mask == 0)
        return self.encoder(sequence_input, src_key_padding_mask=pad_mask)

    def init_weights(self):
        for layer in self.encoder.layers:
            for name, mod in layer.named_modules():
                if isinstance(mod, nn.Linear):
                    nn.init.xavier_uniform_(mod.weight)
                    if mod.bias is not None:
                        nn.init.constant_(mod.bias, 0.0)
                elif isinstance(mod, nn.LayerNorm):
                    nn.init.constant_(mod.weight, 1.0)
                    if mod.bias is not None:
                        nn.init.constant_(mod.bias, 0.0)


### smiles decoder
class SmilesDecoder(nn.Module):
    def __init__(
        self, 
        embed_dim, 
        vocab_size, 
        n_layers, 
        n_heads, 
        pad_token_id=1,
        bos_token_id=0,
        eos_token_id=2,
        dim_feedforward=None, 
        dropout=0.1, 
        activation='gelu',
        validate_n_samples=10,
        predict_n_samples=25
    ):
        super().__init__()
        self.bos_token_id = bos_token_id
        self.pad_token_id = pad_token_id
        self.eos_token_id = eos_token_id
        self.vocab_size = vocab_size
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.embed_dim = embed_dim
        dim_feedforward = dim_feedforward if dim_feedforward is not None else 4 * self.embed_dim
        self.n_samples_per_stage = {'validate': validate_n_samples, 'predict': predict_n_samples}
        
        self.wpe = nn.Embedding(MAX_SMILES_TOKENS + 1, self.embed_dim)
        self.wte = nn.Embedding(vocab_size, self.embed_dim, padding_idx=self.pad_token_id)
        self.decoder = nn.TransformerDecoder(
            decoder_layer=nn.TransformerDecoderLayer(
                d_model=self.embed_dim,
                dim_feedforward=dim_feedforward,
                nhead=self.n_heads,
                dropout=dropout,
                activation=activation,
                batch_first=True,
            ),
            num_layers=self.n_layers,
        )
        self.lm_head = nn.Linear(self.embed_dim, vocab_size, bias=False)
        self.init_weights()

    def init_weights(self):
        torch.nn.init.zeros_(self.lm_head.weight) # zero out classifier weights to start
        torch.nn.init.normal_(self.wte.weight, mean=0.0, std=1.0)
        # if self.wte.weight.device.type == "cuda": # save memory by casting embeddings to bf16
        #     self.wte.to(dtype=torch.bfloat16)
        for layer in self.decoder.layers:
            for name, mod in layer.named_modules():
                if isinstance(mod, nn.Linear):
                    nn.init.xavier_uniform_(mod.weight)
                    if mod.bias is not None:
                        nn.init.constant_(mod.bias, 0.0)
                elif isinstance(mod, nn.LayerNorm):
                    nn.init.constant_(mod.weight, 1.0)
                    if mod.bias is not None:
                        nn.init.constant_(mod.bias, 0.0)

    def forward(self, idx, encoder_outputs, encoder_attention_mask, structure_tokens=None):
        encoder_pad_mask = (encoder_attention_mask == 0)
        softcap = 15
        positions = torch.arange(idx.shape[1], device=idx.device)
        tgt = self.wte(idx) + self.wpe(positions)[None, :, :]
        tgt_mask = torch.triu(torch.ones(idx.shape[1],idx.shape[1], device=idx.device, dtype=torch.bool), diagonal=1)
        x = self.decoder(
            tgt=tgt,
            memory=encoder_outputs,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=(idx == self.pad_token_id),
            memory_key_padding_mask=encoder_pad_mask,
        )
        logits = self.lm_head(x)
        logits = softcap * torch.tanh(logits / softcap)
        logits = logits.float()

        output_dict = {
            'logits': logits,
        }
        if structure_tokens is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                structure_tokens.view(-1),
                ignore_index=self.pad_token_id,
                reduction='mean'
            )
            output_dict['loss'] = loss
        return output_dict
    

    @torch.inference_mode()
    def generate(self, encoder_states, encoder_mask, n_samples=25,
                 max_new_tokens=MAX_SMILES_TOKENS-1, temperature=0.85):
        B = encoder_states.shape[0]
        memory = encoder_states.repeat_interleave(n_samples, dim=0)
        mask = encoder_mask.repeat_interleave(n_samples, dim=0)
        idx = torch.full((B*n_samples,1), self.bos_token_id, device=memory.device, dtype=torch.long)
        done = torch.zeros(B*n_samples, device=memory.device, dtype=torch.bool)
        scores = torch.zeros(B*n_samples, device=memory.device)
        for _ in range(max_new_tokens):
            logits = self.forward(idx, memory, mask)['logits'][:,-1].float()
            logits[:,self.pad_token_id] = -torch.inf
            logits[:,self.bos_token_id] = -torch.inf
            logits[:,tokenizer.token_to_id('<unk>')] = -torch.inf
            if idx.shape[1]==1: logits[:,self.eos_token_id] = -torch.inf
            logp = logits.log_softmax(-1)
            sample_logits = logits / temperature
            sample_logits[done] = -torch.inf
            sample_logits[done,self.eos_token_id] = 0.
            nxt = torch.multinomial(sample_logits.softmax(-1),1)
            increment = logp.gather(1,nxt).squeeze(1)
            scores += torch.where(done,0.,increment)
            idx = torch.cat([idx,nxt],dim=1)
            done |= nxt.squeeze(1)==self.eos_token_id
            if done.all(): break
        scores[~done] = -torch.inf
        return idx.reshape(B,n_samples,-1), scores.reshape(B,n_samples)


In [ ]:
def ranked_candidates(tokens, scores):
    best = {}
    for seq, score in zip(tokens, scores):
        score = float(score)
        if not np.isfinite(score): continue
        text = decode_tokenized_smiles(seq.tolist())
        item = identity(text)
        if item:
            key, smi = item
            if key not in best or score > best[key][0]: best[key]=(score,smi)
    return sorted(((score,key,smi) for key,(score,smi) in best.items()), reverse=True)[:25]

class DeNovoLightningModel(L.LightningModule):
    def __init__(self, kwargs):
        super().__init__()
        self.save_hyperparameters({'kwargs':kwargs})
        self.peak_embedder=PeakEmbedder(**kwargs['peak_embedder'])
        self.spectrum_encoder=SpectrumEncoder(**kwargs['spectrum_encoder'])
        self.smiles_decoder=SmilesDecoder(**kwargs['smiles_decoder'])
        self.optimizer_config=kwargs['optimizer']
    def encode(self,batch):
        return self.spectrum_encoder(self.peak_embedder(batch['mzs'],batch['intensities']),batch['attention_mask'])
    def teacher(self,batch):
        memory=self.encode(batch)
        tokens=batch['structure_tokens']
        output=self.smiles_decoder(tokens[:,:-1],memory,batch['attention_mask'],structure_tokens=tokens[:,1:].contiguous())
        return output,memory
    def training_step(self,batch,batch_idx):
        output,_=self.teacher(batch)
        self.log('train_loss',output['loss'],on_step=True,on_epoch=True,batch_size=len(batch['mzs']))
        return output['loss']
    def validation_step(self,batch,batch_idx):
        output,memory=self.teacher(batch)
        tokens,scores=self.smiles_decoder.generate(memory,batch['attention_mask'],n_samples=25)
        rr=[]
        for ts,ss,label in zip(tokens,scores,batch['smiles']):
            truth=identity(label)[0]
            ranked=ranked_candidates(ts,ss)
            rr.append(next((1./(i+1) for i,(_,key,_) in enumerate(ranked) if key==truth),0.))
        self.log('val_loss',output['loss'],on_epoch=True,batch_size=len(rr))
        self.log('val_mrr25',float(np.mean(rr)),on_epoch=True,batch_size=len(rr),prog_bar=True)
    def configure_optimizers(self):
        opt=AdamW(self.parameters(),lr=self.optimizer_config['lr'],weight_decay=0.01)
        total=max(int(self.trainer.estimated_stepping_batches),1)
        warmup=max(1,int(total*0.05))
        def scale(step):
            if step<warmup:return (step+1)/warmup
            return 0.1+0.9*0.5*(1+math.cos(math.pi*min(1.,(step-warmup)/max(total-warmup,1))))
        return {'optimizer':opt,'lr_scheduler':{'scheduler':torch.optim.lr_scheduler.LambdaLR(opt,scale),'interval':'step'}}

EMBED_DIM, DIM_FEEDFORWARD, DROPOUT = 384, 1536, 0.1
ENCODER_N_HEADS=DECODER_N_HEADS=8
ENCODER_N_LAYERS=DECODER_N_LAYERS=4
ENCODER_ACTIVATION=DECODER_ACTIVATION='gelu'
N_SAMPLES_VAL,N_SAMPLES_PRED,LEARNING_RATE=25,64,2e-4


In [ ]:
model_params = {
    'peak_embedder': {
        'd_model': EMBED_DIM, 
        'dropout': DROPOUT, 
    },
    'spectrum_encoder': {
        'embed_dim': EMBED_DIM,
        'n_heads': ENCODER_N_HEADS,
        'n_layers': ENCODER_N_LAYERS,
        'dim_feedforward': DIM_FEEDFORWARD,
        'dropout': DROPOUT,
        'activation': ENCODER_ACTIVATION,
    },
    'smiles_decoder': {
        'embed_dim': EMBED_DIM, 
        'vocab_size': VOCAB_SIZE, 
        'n_layers': DECODER_N_LAYERS, 
        'n_heads': DECODER_N_HEADS, 
        'pad_token_id': BPE_PAD_ID,
        'bos_token_id': BPE_BOS_ID,
        'eos_token_id': BPE_EOS_ID,
        'dim_feedforward': DIM_FEEDFORWARD, 
        'dropout': DROPOUT, 
        'activation': DECODER_ACTIVATION,
        'validate_n_samples': N_SAMPLES_VAL,
        'predict_n_samples': N_SAMPLES_PRED,
    },
    'optimizer': {
        'lr': LEARNING_RATE,
    },
}
model = DeNovoLightningModel(model_params)

In [ ]:
checkpoint_callback=L.pytorch.callbacks.ModelCheckpoint(
    dirpath=str(WORKING_DIR/'checkpoints'),monitor='val_mrr25',mode='max',
    save_top_k=1,save_last=True,filename='epoch{epoch}-mrr{val_mrr25:.4f}')
trainer=L.Trainer(accelerator='gpu',devices=1,precision=PRECISION,
    max_epochs=8,max_time={'hours':4,'minutes':30},
    callbacks=[checkpoint_callback],logger=False,gradient_clip_val=1.,
    accumulate_grad_batches=2,num_sanity_val_steps=0,log_every_n_steps=50)
trainer.fit(model,datamodule=datamodule)
best=checkpoint_callback.best_model_path
if not best: raise RuntimeError('No validated checkpoint; do not submit an unvalidated model')
model=DeNovoLightningModel.load_from_checkpoint(best,kwargs=model_params,map_location='cpu')
print('Best validation MRR@25:',checkpoint_callback.best_model_score)


## Predict and write a checked submission
Pool candidate structures from up to three spectra per molecule. The highest sequence log probability ranks duplicate candidates. This aggregation and generation temperature are initial settings, not validated optimal choices. A structurally valid nearest-mass training candidate is used only if generation produces no valid output; fallback counts are reported. The time guard stops with an error rather than silently returning a partial submission.

In [ ]:
casmi_test_df=process_spectra(pd.read_parquet(TEST_PATH))
# Prefer peak-rich spectra while preserving different adducts when available.
selected=[]
for mid,g in casmi_test_df.groupby('molecule_id',sort=False):
    g=g.copy();g['peak_count']=g.processed_mzs.map(len)
    g=g.sort_values('peak_count',ascending=False)
    unique_adduct=g.drop_duplicates('adduct')
    chosen=pd.concat([unique_adduct,g]).drop_duplicates('spectrum_id').head(3)
    selected.append(chosen)
casmi_test_df=pd.concat(selected,ignore_index=True)
datamodule.predict_batch_size=1
datamodule.setup('predict')
model=model.cuda().eval()
dtype=torch.bfloat16 if PRECISION=='bf16-mixed' else torch.float16
pooled=defaultdict(dict)
with torch.inference_mode():
    for batch in tqdm(datamodule.predict_dataloader()):
        if time.monotonic()-START>8.5*3600:
            raise TimeoutError('Total budget nearly exhausted; reduce training/generation and rerun')
        mids=batch.pop('molecule_id')
        batch={k:v.cuda(non_blocking=True) for k,v in batch.items()}
        with torch.autocast('cuda',dtype=dtype):
            memory=model.encode(batch)
            tokens,scores=model.smiles_decoder.generate(memory,batch['attention_mask'],n_samples=64)
        for mid,ts,ss in zip(mids,tokens.cpu(),scores.cpu()):
            for score,key,smi in ranked_candidates(ts,ss):
                if key not in pooled[mid] or score>pooled[mid][key][0]:pooled[mid][key]=(score,smi)
sample=pd.read_csv(SAMPLE_SUBMISSION_PATH,dtype={'molecule_id':str})
assert set(sample.molecule_id)==set(casmi_test_df.molecule_id)

from rdkit.Chem import Descriptors
DELTA={'[M+H]+':1.007276466621,'[M-H]-':-1.007276466621,'[M+Na]+':22.989218,
       '[M+K]+':38.963158,'[M+NH4]+':18.033823,'[M+Cl]-':34.969402,'[M+CH2O2-H]-':44.998201}
fallback_pool=[]
for s in casmi_train_df.normalized_smiles.unique():
    mol=Chem.MolFromSmiles(s)
    if mol is not None and Chem.GetFormalCharge(mol)==0 and '.' not in s:
        fallback_pool.append((Descriptors.ExactMolWt(mol),s))
fallback_pool.sort()
fm=np.array([m for m,_ in fallback_pool])
fallback_count=0
def guesses(mid):
    global fallback_count
    ranked=sorted(pooled[mid].values(),reverse=True)[:25]
    if ranked:return ';'.join(s for _,s in ranked)
    fallback_count+=1
    group=casmi_test_df[casmi_test_df.molecule_id==mid]
    neutral=[r.precursor_mz-DELTA[r.adduct] for r in group.itertuples() if r.adduct in DELTA]
    if not neutral or not len(fm):raise ValueError(f'No valid prediction or mass fallback for {mid}')
    return fallback_pool[int(np.argmin(abs(fm-np.median(neutral))))][1]
submission=sample[['molecule_id']].copy()
submission['smiles']=submission.molecule_id.map(guesses)
assert not submission.isna().any().any() and not submission.molecule_id.duplicated().any()
for value in submission.smiles:
    items=[identity(s) for s in value.split(';')]
    assert 1<=len(items)<=25 and all(items)
    assert len({v[0] for v in items})==len(items)
submission.to_csv(WORKING_DIR/'submission.csv',index=False)
print('Wrote',len(submission),'molecules; fallback molecules:',fallback_count)
print('Total elapsed hours:',(time.monotonic()-START)/3600)
